In [1]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="ollama",
    model="qwen3:8b"
)

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read the latest email."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [4]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my latest email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Majid, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, Majid Jr."
    },
    config=config
)

In [5]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'This '
                                                                          'is '
                                                                          'an '
                                                                          'automatic '
                                                                          'reply '
                                                                          'to '
                                                                          'your '
                                                                          'latest '
                                                                          'email. '
                                                                          'Please '
                                                                          'let '
                                                                          'me '
             

In [6]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'This is an automatic reply to your latest email. Please let me know if you need further assistance.'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'This is an automatic reply to your latest email. Please let me know if you need further assistance.'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='80e3c436d3fa2e37051f913394052a7f')]


In [7]:
print(response["messages"][-1].content)

<think>
Okay, the user wants me to read their latest email and send a response immediately in the same thread. Let me check the tools available.

First, there's the read_email function. It doesn't require any parameters, so I can call that to get the latest email. Once I have the email details, I need to prepare a reply. The send_email function requires a body parameter. Wait, the user didn't specify the content of the response, so maybe I should assume a standard reply or ask for more details? But the user said to send the reply now, so perhaps they expect an automatic response. Since the body is required, I'll need to generate a placeholder reply. Let me structure the tool calls: first read the email, then send the reply with a sample body. But the functions don't have parameters for the reply content, so maybe the user expects me to use the read_email's content to form the reply. However, since I can't access the email content here, I'll have to proceed with the functions as per the

In [8]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

This is an automatic reply to your latest email. Please let me know if you need further assistance.


## Approve

In [9]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Majid, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, Majid Jr.',
 'messages': [HumanMessage(content='Please read my latest email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='1c84407a-3004-43eb-8e9c-f730835f5917'),
              AIMessage(content='<think>\nOkay, the user wants me to read their latest email and send a response immediately in the same thread. Let me check the tools available.\n\nFirst, there\'s the read_email function. It doesn\'t require any parameters, so I can call that to get the latest email. Once I have the email details, I need to prepare a response. But wait, the send_email function requires a body parameter. The user didn\'t specify the content of the reply, so maybe I should ask for clarification. However, the user said to send the reply now, so perhaps they expect an automated response. Since the body is required, maybe I should use 

## Reject

In [10]:
from langgraph.types import Command

response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "please sign off - Your leader, Majid."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Sure, '
                                                                          "let's "
                                                                          'reschedule. '
                                                                          'What '
                                                                          'time '
                                                                          'works '
                                                                          'better '
                                                                          'for '
                                                                          'you? '
                                                                          'Best, '
                                                                          'Majid'},
                                                         'description': 'Tool '
    

In [11]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Sure, let's reschedule. What time works better for you? Best, Majid


## Edit

In [9]:
from langgraph.types import Command

response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Majid, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, Majid Jr.',
 'messages': [HumanMessage(content='Please read my latest email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='9b0b7931-130b-420e-b4c8-5d3a1d64caba'),
              AIMessage(content="<think>\nOkay, the user wants me to read their latest email and send a response immediately in the same thread. Let me check the tools available.\n\nFirst, there's the read_email function. It doesn't require any parameters, so I can call that to get the latest email. Once I have the email details, I need to prepare a reply. The send_email function requires a body parameter. Wait, the user didn't specify the content of the response, so maybe I should assume a standard reply or ask for more details? But the user said to send the reply now, so perhaps they expect an automatic response. Since the body is required, I'll 